# 🧠 Stage 1: Code + Reasoning Fine-Tuning
> Fine-tune Qwen2.5-Coder-7B-Instruct on combined code generation and reasoning data.

**Base Model:** `Qwen/Qwen2.5-Coder-7B-Instruct`
**Method:** QLoRA (4-bit quantization + LoRA adapters)
**Dataset:** ~340K code + reasoning samples from Stage 0
**GPU Required:** A100 40/80GB or H100 (Colab Pro)
**Estimated Time:** 8-12 hours (A100), 5-8 hours (H100)

### Why QLoRA?
- Full fine-tuning of 7B model needs ~60GB VRAM (FP16) — too much for single GPU
- QLoRA achieves ~97% of full fine-tuning quality with ~6-8GB VRAM
- LoRA rank 32 provides good capacity for multi-task learning


In [ ]:
# Install Unsloth for 2x faster training + 60% less memory
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q wandb datasets

In [ ]:
import torch
import json
import os
from unsloth import FastLanguageModel
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments

# Verify GPU
print(f"🖥️ GPU: {torch.cuda.get_device_name(0)}")
print(f"💾 VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
print(f"🔥 CUDA: {torch.version.cuda}")

In [ ]:
# Configuration
MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct"
MAX_SEQ_LENGTH = 4096  # Balance between context and memory
LORA_RANK = 32         # Higher rank = more capacity for multi-task
LORA_ALPHA = 32        # Common to match rank
OUTPUT_DIR = "outputs/stage1_code_reasoning"

# Training hyperparameters (research-backed defaults)
BATCH_SIZE = 2                  # Per-device batch size
GRADIENT_ACCUMULATION = 8       # Effective batch = 16
LEARNING_RATE = 2e-4            # Standard for QLoRA
NUM_EPOCHS = 2                  # 2 epochs for large datasets
WARMUP_RATIO = 0.05             # 5% warmup
WEIGHT_DECAY = 0.01
LR_SCHEDULER = "cosine"

print("✅ Config loaded")
print(f"   Effective batch size: {BATCH_SIZE * GRADIENT_ACCUMULATION}")

In [ ]:
# Load model with Unsloth (2x faster, 60% less memory)
print("📥 Loading model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,              # Auto-detect (bfloat16 on A100/H100)
    load_in_4bit=True,       # QLoRA 4-bit quantization
)

# Add LoRA adapters
print("🔧 Adding LoRA adapters...")
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",    # Attention
        "gate_proj", "up_proj", "down_proj",         # MLP
    ],
    lora_dropout=0.05,       # Small dropout for regularization
    bias="none",
    use_gradient_checkpointing="unsloth",  # 30% more memory efficient
    random_state=42,
)

# Print trainable parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"\n📊 Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

In [ ]:
# Load prepared dataset from Stage 0
print("📥 Loading prepared dataset...")

# If running after notebook 01, data is in prepared_data/
# If data isn't local, you can upload or mount from Google Drive
DATA_PATH = "prepared_data/stage1_code_reasoning.jsonl"

if not os.path.exists(DATA_PATH):
    print("⚠️ Dataset not found locally. Attempting to download from HF...")
    print("   Please run notebook 01 first, or upload the prepared data.")
    # Fallback: you could host prepared data on HF Hub
    raise FileNotFoundError(f"{DATA_PATH} not found. Run notebook 01 first.")

# Load JSONL
samples = []
with open(DATA_PATH, "r") as f:
    for line in f:
        samples.append(json.loads(line))

print(f"✅ Loaded {len(samples):,} samples")

# Convert messages to tokenizer chat template format
def format_for_training(sample):
    """Apply Qwen2.5-Coder chat template to messages."""
    text = tokenizer.apply_chat_template(
        sample["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": text}

# Create HF Dataset
dataset = Dataset.from_list(samples)
dataset = dataset.map(
    format_for_training,
    remove_columns=dataset.column_names,
    num_proc=4,
    desc="Formatting"
)

print(f"✅ Formatted dataset: {len(dataset):,} samples")
print(f"   Example length: {len(dataset[0]['text'])} chars")

In [ ]:
# Optional: Login to W&B for experiment tracking
# import wandb
# wandb.login()
# os.environ["WANDB_PROJECT"] = "code-llm-forge"

# Initialize trainer
print("🏋️ Setting up trainer...")
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=4,
    packing=True,              # Pack short examples together for efficiency
    args=TrainingArguments(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type=LR_SCHEDULER,
        warmup_ratio=WARMUP_RATIO,
        weight_decay=WEIGHT_DECAY,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        save_strategy="steps",
        save_steps=500,
        save_total_limit=3,
        optim="adamw_8bit",     # Memory-efficient optimizer
        seed=42,
        report_to="none",       # Change to "wandb" if using W&B
        dataloader_num_workers=2,
    ),
)

print(f"\n📊 Training plan:")
print(f"   Total samples: {len(dataset):,}")
print(f"   Batch size (effective): {BATCH_SIZE * GRADIENT_ACCUMULATION}")
print(f"   Steps per epoch: ~{len(dataset) // (BATCH_SIZE * GRADIENT_ACCUMULATION):,}")
print(f"   Total steps: ~{NUM_EPOCHS * len(dataset) // (BATCH_SIZE * GRADIENT_ACCUMULATION):,}")

In [ ]:
# 🚀 TRAIN!
print("🚀 Starting Stage 1 training...")
print("=" * 60)
trainer_stats = trainer.train()

print("\n" + "=" * 60)
print("✅ Stage 1 training complete!")
print(f"   Total time: {trainer_stats.metrics['train_runtime']/3600:.1f} hours")
print(f"   Final loss: {trainer_stats.metrics['train_loss']:.4f}")

In [ ]:
# Save the LoRA adapter
print("💾 Saving Stage 1 adapter...")
model.save_pretrained(f"{OUTPUT_DIR}/lora_adapter")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/lora_adapter")
print(f"✅ Saved to {OUTPUT_DIR}/lora_adapter")

# Optional: Merge and save full model (uses more disk but easier to use)
print("\n🔀 Merging adapter into full model...")
merged_model = model.merge_and_unload()
merged_model.save_pretrained(f"{OUTPUT_DIR}/merged_model")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/merged_model")
print(f"✅ Merged model saved to {OUTPUT_DIR}/merged_model")
print("\n🎉 Stage 1 complete! Proceed to notebook 03 for tool-calling training.")